In [1]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "country": ["Thailand", "Thailand", "Singapore", "Malaysia"]
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 1, 2, 3, 5],
    "revenue": [100, 150, 80, 200, 50]
})

In [4]:
customers

,customer_id,country
0,1,Thailand
1,2,Thailand
2,3,Singapore
3,4,Malaysia


In [3]:
customers
orders

,order_id,customer_id,revenue
0,101,1,100
1,102,1,150
2,103,2,80
3,104,3,200
4,105,5,50


In [ ]:
orders_with_country = orders.merge(
    customers, 
    on="customer_id", 
    how="left",
    validate="m:1"
)

orders_with_country

,order_id,customer_id,revenue,country
0,101,1,100,Thailand
1,102,1,150,Thailand
2,103,2,80,Thailand
3,104,3,200,Singapore
4,105,5,50,NaN


In [6]:
matched_order = orders.merge(
    customers, 
    on="customer_id",
    how="inner",
    validate="m:1"
)

matched_order

,order_id,customer_id,revenue,country
0,101,1,100,Thailand
1,102,1,150,Thailand
2,103,2,80,Thailand
3,104,3,200,Singapore


In [7]:
print(orders_with_country["revenue"].sum())
print(matched_order["revenue"].sum())

580
530


In [9]:
customers_with_orders = customers.merge(
    orders,
    on="customer_id",
    how="left",
    validate="1:m"
)

customers_with_orders

,customer_id,country,order_id,revenue
0,1,Thailand,101.0,100.0
1,1,Thailand,102.0,150.0
2,2,Thailand,103.0,80.0
3,3,Singapore,104.0,200.0
4,4,Malaysia,NaN,NaN


In [24]:
customer_order_check = customers.merge(
    orders,
    on="customer_id",
    how="left",
    indicator=True,
    validate="1:m"
)

customers_without_orders = customer_order_check.loc[
    customer_order_check["_merge"] == "left_only", 
    ["customer_id", "country", "order_id", "revenue"]
]

print(customer_order_check)
print(customers_without_orders)

   customer_id    country  order_id  revenue     _merge
0            1   Thailand     101.0    100.0       both
1            1   Thailand     102.0    150.0       both
2            2   Thailand     103.0     80.0       both
3            3  Singapore     104.0    200.0       both
4            4   Malaysia       NaN      NaN  left_only
   customer_id   country  order_id  revenue
4            4  Malaysia       NaN      NaN


In [33]:
order_customer_check = orders.merge(
    customers, 
    on="customer_id",
    how="left",
    validate="m:1",
    indicator=True
)

orders_without_customers = order_customer_check.loc[
    order_customer_check["_merge"] == "left_only", 
    ~order_customer_check.columns.isin(["_merge", "country"])
]

orders_without_customers

,order_id,customer_id,revenue
4,105,5,50


In [26]:
orders

,order_id,customer_id,revenue
0,101,1,100
1,102,1,150
2,103,2,80
3,104,3,200
4,105,5,50


In [27]:
customers

,customer_id,country
0,1,Thailand
1,2,Thailand
2,3,Singapore
3,4,Malaysia


In [34]:
import sqlite3

connection = sqlite3.connect(":memory:")

customers.to_sql("customers", connection, index=False)
orders.to_sql("orders", connection, index=False)


5

In [ ]:
query = """
    select * from orders as o 
    left join customers c on o.customer_id = c.customer_id 
    where c.customer_id is null
"""

pd.read_sql_query(query, connection)

,order_id,customer_id,revenue,customer_id,country
0,105,5,50,None,None


In [38]:
query = """
select * from customers c left join orders o on c.customer_id = o.customer_id
where o.customer_id is null
"""
pd.read_sql_query(query, connection)


,customer_id,country,order_id,customer_id,revenue
0,4,Malaysia,None,None,None


In [43]:
query="""
    select coalesce(c.country, 'Unknown') as country,
    count(o.order_id) as order_count, 
    sum(o.revenue) as total_revenue

    from orders o left join customers c on o.customer_id = c.customer_id
    group by coalesce(c.country, 'unknown')
    having sum(o.revenue) >= 200
    order by total_revenue desc

"""

pd.read_sql_query(query, connection)

,country,order_count,total_revenue
0,Thailand,3,330
1,Singapore,1,200


In [ ]:
query = """ 

With ranked_orders as (
    select order_id, customer_id, revenue, row_number() over (
        partition by customer_id 
        order by revenue desc, order_id asc
    ) as rn
    from orders
)

select order_id, customer_id, revenue from ranked_orders
where rn = 1 order by customer_id

"""

pd.read_sql_query(query, connection)


,order_id,customer_id,revenue
0,102,1,150
1,103,2,80
2,104,3,200
3,105,5,50


In [56]:
query = """ 
    with ranked_orders as (
        select order_id, customer_id, revenue, row_number() over(
            partition by customer_id order by revenue asc, order_id asc
        ) as rn 
        from orders
    )
    select order_id, customer_id, revenue from ranked_orders
    where rn = 1 order by customer_id

    """

pd.read_sql_query(query, connection)

,order_id,customer_id,revenue
0,101,1,100
1,103,2,80
2,104,3,200
3,105,5,50


In [54]:
query = """
    select * from orders
"""

pd.read_sql_query(query, connection)

,order_id,customer_id,revenue
0,101,1,100
1,102,1,150
2,103,2,80
3,104,3,200
4,105,5,50


In [45]:
query = """
    select * from customers
"""

pd.read_sql_query(query, connection)

,customer_id,country
0,1,Thailand
1,2,Thailand
2,3,Singapore
3,4,Malaysia


In [ ]:
query = """ 

With ranked_orders as (
    select order_id, customer_id, revenue, row_number() over (
        partition by customer_id 
        order by revenue desc, order_id asc
    ) as rn
    from orders
)

select order_id, customer_id, revenue from ranked_orders
where rn = 1 order by customer_id

"""

pd.read_sql_query(query, connection)


,order_id,customer_id,revenue
0,102,1,150
1,103,2,80
2,104,3,200
3,105,5,50


In [ ]:
query="""
    select coalesce(c.country, 'Unknown') as country,
    count(o.order_id) as order_count, 
    sum(o.revenue) as total_revenue

    from orders o left join customers c on o.customer_id = c.customer_id
    group by coalesce(c.country, 'unknown')
    having sum(o.revenue) >= 200
    order by total_revenue desc

"""

pd.read_sql_query(query, connection)

,country,order_count,total_revenue
0,Thailand,3,330
1,Singapore,1,200


In [59]:
query = """
    select order_id, customer_id, revenue, sum(revenue) over (
        partition by customer_id
    ) as customer_total_revenue

    from orders 
    order by customer_id, order_id
"""

pd.read_sql_query(query, connection)

,order_id,customer_id,revenue,customer_total_revenue
0,101,1,100,250
1,102,1,150,250
2,103,2,80,80
3,104,3,200,200
4,105,5,50,50


In [58]:
orders

,order_id,customer_id,revenue
0,101,1,100
1,102,1,150
2,103,2,80
3,104,3,200
4,105,5,50


In [60]:
orders_with_totals = orders.copy()

orders_with_totals["customer_total_revenue"] = (
    orders_with_totals.groupby("customer_id")["revenue"].transform("sum")
)

orders_with_totals

,order_id,customer_id,revenue,customer_total_revenue
0,101,1,100,250
1,102,1,150,250
2,103,2,80,80
3,104,3,200,200
4,105,5,50,50


In [62]:
orders_with_totals["order_share_pct"] = orders_with_totals["revenue"] / orders_with_totals["customer_total_revenue"] * 100

orders_with_totals

,order_id,customer_id,revenue,customer_total_revenue,order_share_pct
0,101,1,100,250,40.0
1,102,1,150,250,60.0
2,103,2,80,80,100.0
3,104,3,200,200,100.0
4,105,5,50,50,100.0


In [70]:
query = """ 
    select c.customer_id, c.country, count(o.order_id) as order_count, coalesce(sum(o.revenue), 0) as total_revenue
    from customers c left join orders o on c.customer_id = o.customer_id
    group by c.customer_id, c.country
    order by total_revenue desc
"""


pd.read_sql_query(query, connection)


,customer_id,country,order_count,total_revenue
0,1,Thailand,2,250
1,3,Singapore,1,200
2,2,Thailand,1,80
3,4,Malaysia,0,0


In [69]:
customers

,customer_id,country
0,1,Thailand
1,2,Thailand
2,3,Singapore
3,4,Malaysia


,order_id,customer_id,revenue
0,101,1,100
1,102,1,150
2,103,2,80
3,104,3,200
4,105,5,50


In [71]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [ ]:
customers_summary = customers.merge(
    orders,
    on="customer_id",
    how="left"
    validate=""
)

In [78]:
help(orders.agg)

Help on method aggregate in module pandas.core.frame:

aggregate(func=None, axis: 'Axis' = 0, *args, **kwargs) method of pandas.core.frame.DataFrame instance
    Aggregate using one or more operations over the specified axis.
    
    Parameters
    ----------
    func : function, str, list or dict
        Function to use for aggregating the data. If a function, must either
        work when passed a DataFrame or when passed to DataFrame.apply.
    
        Accepted combinations are:
    
        - function
        - string function name
        - list of functions and/or function names, e.g. ``[np.sum, 'mean']``
        - dict of axis labels -> functions, function names or list of such.
    axis : {0 or 'index', 1 or 'columns'}, default 0
            If 0 or 'index': apply function to each column.
            If 1 or 'columns': apply function to each row.
    *args
        Positional arguments to pass to `func`.
    **kwargs
        Keyword arguments to pass to `func`.
    
    Return

In [85]:
order_summary = orders.groupby("customer_id", as_index=False).agg(
    order_counts=("order_id", "nunique"),
    total_revenue=("revenue", "sum")
)



In [90]:
customer_summary = customers.merge(
    order_summary,
    on="customer_id",
    how="left",
    validate="1:1"
)

In [92]:
customer_summary[["order_counts", "total_revenue"]] = (
    customer_summary[["order_counts", "total_revenue"]].fillna(0)
)

customer_summary

,customer_id,country,order_counts,total_revenue
0,1,Thailand,2.0,250.0
1,2,Thailand,1.0,80.0
2,3,Singapore,1.0,200.0
3,4,Malaysia,0.0,0.0


In [97]:
help(orders.sort_values)

Help on method sort_values in module pandas.core.frame:

sort_values(by: 'IndexLabel', *, axis: 'Axis' = 0, ascending: 'bool | list[bool] | tuple[bool, ...]' = True, inplace: 'bool' = False, kind: 'SortKind' = 'quicksort', na_position: 'str' = 'last', ignore_index: 'bool' = False, key: 'ValueKeyFunc | None' = None) -> 'DataFrame | None' method of pandas.core.frame.DataFrame instance
    Sort by the values along either axis.
    
    Parameters
    ----------
    by : str or list of str
        Name or list of names to sort by.
    
        - if `axis` is 0 or `'index'` then `by` may contain index
          levels and/or column labels.
        - if `axis` is 1 or `'columns'` then `by` may contain column
          levels and/or index labels.
    axis : "{0 or 'index', 1 or 'columns'}", default 0
         Axis to be sorted.
    ascending : bool or list of bool, default True
         Sort ascending vs. descending. Specify list for multiple sort
         orders.  If this is a list of bools,

In [102]:
customer_summary["average_value"] = (customer_summary["total_revenue"]
/ customer_summary["order_counts"].where(customer_summary["order_counts"] > 0))

customer_summary.sort_values("total_revenue", ascending=False)

,customer_id,country,order_counts,total_revenue,average_value
0,1,Thailand,2.0,250.0,125.0
2,3,Singapore,1.0,200.0,200.0
1,2,Thailand,1.0,80.0,80.0
3,4,Malaysia,0.0,0.0,NaN


In [87]:
customers

,customer_id,country
0,1,Thailand
1,2,Thailand
2,3,Singapore
3,4,Malaysia


In [79]:
orders

,order_id,customer_id,revenue
0,101,1,100
1,102,1,150
2,103,2,80
3,104,3,200
4,105,5,50


In [103]:
country_summary = customer_summary.groupby("country").agg(
    customers=("customer_id", "nunique"),
    orders=("order_counts", "sum"),
    revenue=("total_revenue", "sum")
)

country_summary

,customers,orders,revenue
country,,,
Malaysia,1,0.0,0.0
Singapore,1,1.0,200.0
Thailand,2,3.0,330.0


In [ ]:
orders_with_country.pivot_table(
    index="country",  # row labels
    columns="customer_id", # column labels
    values="revenue",  # values to summarise
    aggfunc="sum",  # calculation within each combination
    fill_value=0   # display empty cominations as zero
)

customer_id,1,2,3
country,,,
Singapore,0,0,200
Thailand,250,80,0


In [105]:
orders_with_country.pivot_table(
    index="country",
    columns="customer_id",
    values="order_id",
    aggfunc="nunique",
    fill_value=0
)

customer_id,1,2,3
country,,,
Singapore,0,0,1
Thailand,2,1,0
